# GateSense 01: train the number-plate detector (YOLOv8n vs YOLOv8s)

Runs on **Kaggle** (recommended) or **Google Colab**, with a GPU. It builds a training set from public Indian number-plate datasets, fine-tunes **YOLOv8n** and **YOLOv8s**, scores both on a **separate test dataset from a different source**, picks one **by the numbers**, exports it to **ONNX** and writes a results report.

**On Kaggle:** *Settings* (right panel) → *Accelerator* → **GPU T4 x2**, and *Internet* → **On** (needs a phone-verified account). Then *Run all*, or *Save Version → Save & Run All* to run it in the background. Results appear under **Output** (`/kaggle/working`).

**On Colab:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Results download as a zip at the end.

### Data (default `DATASET_SOURCE = "kaggle_public"`, no API key needed)
Only datasets with a clear open licence are used, so the trained model can be published:

| Dataset | Licence | Used for |
|---|---|---|
| [deepakat002/indian-vehicle-number-plate-yolo-annotation](https://www.kaggle.com/datasets/deepakat002/indian-vehicle-number-plate-yolo-annotation) | CC0 | train / validation (whole videos per split) |
| [kedarsai/indian-license-plates-with-labels](https://www.kaggle.com/datasets/kedarsai/indian-license-plates-with-labels) | CC0 | train / validation (plate crops dropped) |
| [tkm22092/indian-number-plate-images](https://www.kaggle.com/datasets/tkm22092/indian-number-plate-images) | CC0 | train / validation (gate camera photos) |
| [dataclusterlabs/indian-number-plates-dataset](https://www.kaggle.com/datasets/dataclusterlabs/indian-number-plates-dataset) | CC0 | train / validation (VOC labels converted) |
| [paneraghanshyam/gujarat-vehicle-number-plates-yolo-ready](https://www.kaggle.com/datasets/paneraghanshyam/gujarat-vehicle-number-plates-yolo-ready) | Apache 2.0 | **test only** |

Exact duplicates are removed across all sources (so no test photo is also a training photo), and boxes covering more than half the picture (already-cropped plates) are dropped. Other choices: `"roboflow"` (needs a free Roboflow key saved as the secret `ROBOFLOW_API_KEY`: Kaggle *Add-ons → Secrets*, Colab key icon) or `"folder"` (your own photos in YOLO format, see `anpr/DATASET.md`).

In [ ]:
# --- Settings (the defaults work; change only if you know why) ---
DATASET_SOURCE = "kaggle_public"  # "kaggle_public", "roboflow" or "folder"
# "roboflow": any YOLO-format plate set on Roboflow Universe
ROBOFLOW_WORKSPACE = "licenceplatedetection-i880o"
ROBOFLOW_PROJECT = "indian-plate-detection-object-detection"
ROBOFLOW_VERSION = None  # None = latest version
# "folder": data.yaml + train/valid(/test) in YOLO format (Kaggle: an attached dataset under
# /kaggle/input/...; Colab: a Google Drive folder, mounted automatically)
FOLDER_DATASET = "/content/drive/MyDrive/gatesense/detector_dataset"

CANDIDATES = ["yolov8n.pt", "yolov8s.pt"]  # pretrained on COCO, fine-tuned here
EPOCHS = 60
IMAGE_SIZE = 640
# Pick the bigger model only if it is clearly more accurate (mAP50-95 points on the test set).
MIN_GAIN_FOR_BIGGER_MODEL = 0.02

In [ ]:
import os, subprocess, sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "google.colab" in sys.modules
PLATFORM = "Kaggle" if ON_KAGGLE else "Google Colab" if ON_COLAB else "local"
# Results go to OUT (Kaggle shows /kaggle/working as the notebook's Output); bulky scratch
# files (datasets, training runs) stay out of it.
OUT = Path("/kaggle/working/gatesense_results") if ON_KAGGLE else Path("/content/gatesense_results") if ON_COLAB else Path("gatesense_results")
SCRATCH = Path("/tmp/gatesense") if ON_KAGGLE else Path("/content") if ON_COLAB else Path("scratch")
OUT.mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True)

print(PLATFORM, sys.version.split()[0])
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "NO GPU: turn on a GPU accelerator (training on CPU takes hours)")
packages = ["ultralytics>=8.3", "onnx", "onnxslim", "onnxruntime", "kagglehub"]
if DATASET_SOURCE == "roboflow":
    packages.append("roboflow")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


def secret(name):
    """A key from the platform's secret store: never typed into a cell."""
    if ON_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    return os.environ[name]

In [ ]:
# --- Build one clean dataset from the public Kaggle sources ---
import hashlib, random, shutil, xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")
MAX_BOX_AREA = 0.5  # a box covering over half the picture = an already-cropped plate, not a scene


def _image_for(label, by_stem):
    """The photo a YOLO label file belongs to: same folder, or the matching images/ folder."""
    dirs = [label.parent]
    parts = list(label.parent.parts)
    if "labels" in parts:
        i = len(parts) - 1 - parts[::-1].index("labels")
        dirs.append(Path(*parts[:i], "images", *parts[i + 1:]))
    for d in dirs:
        for ext in IMAGE_EXTS:
            if (d / (label.stem + ext)).exists():
                return d / (label.stem + ext)
    same_name = by_stem.get(label.stem, [])
    return same_name[0] if len(same_name) == 1 else None


def _yolo_items(root, by_stem):
    for label in root.rglob("*.txt"):
        if label.name.lower() in ("classes.txt", "class.txt", "readme.txt"):
            continue
        image = _image_for(label, by_stem)
        if image is None:
            continue
        boxes = []
        for line in label.read_text(errors="ignore").splitlines():
            parts = line.split()
            if len(parts) >= 5:
                try:
                    boxes.append(tuple(float(v) for v in parts[1:5]))
                except ValueError:
                    pass
        yield image, boxes


def _voc_items(root, by_stem, names):
    for xml_file in root.rglob("*.xml"):
        try:
            tree = ET.parse(xml_file).getroot()
        except ET.ParseError:
            continue
        if tree.tag != "annotation":
            continue
        stem = Path(tree.findtext("filename") or xml_file.stem).stem
        image = by_stem.get(stem, [None])[0] if len(by_stem.get(stem, [])) == 1 else None
        width, height = float(tree.findtext("size/width") or 0), float(tree.findtext("size/height") or 0)
        if image is None or not width or not height:
            continue
        boxes = []
        for obj in tree.iter("object"):
            name = (obj.findtext("name") or "").strip()
            names[name] += 1
            if "plate" not in name.lower() and "number" not in name.lower():
                continue  # e.g. single characters in an OCR set
            x1, y1, x2, y2 = (float(obj.findtext(f"bndbox/{k}")) for k in ("xmin", "ymin", "xmax", "ymax"))
            boxes.append(((x1 + x2) / 2 / width, (y1 + y2) / 2 / height, (x2 - x1) / width, (y2 - y1) / height))
        yield image, boxes


def collect(root):
    """Every (image, plate boxes) pair in a downloaded dataset, YOLO or Pascal VOC labels."""
    by_stem = defaultdict(list)
    for p in root.rglob("*"):
        if p.suffix in IMAGE_EXTS:
            by_stem[p.stem].append(p)
    names = Counter()
    items = list(_yolo_items(root, by_stem)) + list(_voc_items(root, by_stem, names))
    return items, names


def group_of(slug, image):
    """Pictures that are near-copies of each other (frames of one video) stay in one split."""
    return f"{slug}:{image.parent.name}" if "vid-" in image.parent.name else f"{slug}:{image.stem}"


def build(sources, out_dir, val_fraction=0.15, seed=0, download=None):
    rng = random.Random(seed)
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    seen, stats, pool, test = set(), [], [], []
    for slug, licence, role in sources:
        root = Path(download(slug))
        items, voc_names = collect(root)
        kept = dupes = crops = empty = 0
        for image, boxes in items:
            digest = hashlib.md5(image.read_bytes()).hexdigest()
            if digest in seen:
                dupes += 1
                continue
            seen.add(digest)
            boxes = [b for b in boxes if 0 < b[2] <= 1 and 0 < b[3] <= 1]
            if not boxes:
                empty += 1
                continue
            if max(w * h for _, _, w, h in boxes) > MAX_BOX_AREA:
                crops += 1
                continue
            entry = (image, boxes, group_of(slug, image))
            (test if role == "test" else pool).append(entry)
            kept += 1
        stats.append({"source": slug, "licence": licence, "role": role, "found": len(items),
                      "kept": kept, "duplicates": dupes, "plate_crops": crops, "no_boxes": empty,
                      "voc_classes": dict(voc_names)})
    groups = sorted({g for _, _, g in pool})
    rng.shuffle(groups)
    val_groups = set(groups[: max(1, round(len(groups) * val_fraction))])
    splits = {"train": [e for e in pool if e[2] not in val_groups],
              "valid": [e for e in pool if e[2] in val_groups], "test": test}
    for split, entries in splits.items():
        (out_dir / split / "images").mkdir(parents=True, exist_ok=True)
        (out_dir / split / "labels").mkdir(parents=True, exist_ok=True)
        for n, (image, boxes, _) in enumerate(entries):
            name = f"{n:05d}"
            shutil.copy(image, out_dir / split / "images" / (name + image.suffix.lower()))
            (out_dir / split / "labels" / (name + ".txt")).write_text(
                "".join(f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n" for x, y, w, h in boxes))
    return stats, {k: len(v) for k, v in splits.items()}

In [ ]:
import shutil

KAGGLE_SOURCES = [
    ("deepakat002/indian-vehicle-number-plate-yolo-annotation", "CC0-1.0", "trainval"),
    ("kedarsai/indian-license-plates-with-labels", "CC0-1.0", "trainval"),
    ("tkm22092/indian-number-plate-images", "CC0-1.0", "trainval"),
    ("dataclusterlabs/indian-number-plates-dataset", "CC0-1.0", "trainval"),
    ("paneraghanshyam/gujarat-vehicle-number-plates-yolo-ready", "Apache-2.0", "test"),
]
dataset_dir = SCRATCH / "dataset"
source_stats = []
if DATASET_SOURCE == "kaggle_public":
    import kagglehub
    source_stats, built_counts = build(KAGGLE_SOURCES, dataset_dir, download=kagglehub.dataset_download)
    for s in source_stats:
        print(s)
    print(built_counts)
    source = "public Kaggle datasets (CC0 / Apache 2.0), test set from a separate source"
elif DATASET_SOURCE == "roboflow":
    from roboflow import Roboflow
    project = Roboflow(api_key=secret("ROBOFLOW_API_KEY")).workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = ROBOFLOW_VERSION or max(int(str(v.version).rstrip("/").split("/")[-1]) for v in project.versions())
    dataset_dir = Path(project.version(version).download("yolov8", location=str(dataset_dir)).location)
    source = f"Roboflow `{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}` v{version}"
elif DATASET_SOURCE == "folder":
    if ON_COLAB and FOLDER_DATASET.startswith("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    shutil.copytree(FOLDER_DATASET, dataset_dir, dirs_exist_ok=True)
    source = f"own photos (`{FOLDER_DATASET}`)"
else:
    raise ValueError("DATASET_SOURCE must be 'kaggle_public', 'roboflow' or 'folder'")
print(sorted(p.name for p in dataset_dir.iterdir()))

### Make it a clean one-class dataset
Datasets name the class differently (`plate`, `number_plate`, `License_Plate`…) and some have extra classes. Every box becomes one class, `plate`, and `data.yaml` gets absolute paths.

In [ ]:
import yaml

splits = {name: dataset_dir / name / "images" for name in ("train", "valid", "test")}
splits = {name: path for name, path in splits.items() if path.exists() and any(path.iterdir())}
assert "train" in splits and "valid" in splits, f"expected train/ and valid/ folders, got {list(splits)}"

boxes = 0
for split in splits:
    for label_file in (dataset_dir / split / "labels").glob("*.txt"):
        lines = [l.split() for l in label_file.read_text().splitlines() if l.strip()]
        label_file.write_text("".join("0 " + " ".join(parts[1:5]) + "\n" for parts in lines if len(parts) >= 5))
        boxes += len(lines)

data_yaml = dataset_dir / "gatesense_plate.yaml"
data_yaml.write_text(yaml.safe_dump({
    "path": str(dataset_dir), "train": "train/images", "val": "valid/images",
    **({"test": "test/images"} if "test" in splits else {}),
    "nc": 1, "names": ["plate"],
}))
counts = {split: len(list(path.glob("*"))) for split, path in splits.items()}
print("images per split:", counts, "| plate boxes:", boxes)

In [ ]:
import json, time
from ultralytics import YOLO

EVAL_SPLIT = "test" if "test" in splits else "val"  # never score on the images we trained on
results = []
for weights in CANDIDATES:
    name = Path(weights).stem + "_plate"
    started = time.time()
    model = YOLO(weights)
    model.train(data=str(data_yaml), epochs=EPOCHS, imgsz=IMAGE_SIZE, patience=15, seed=0,
                device=0 if gpu else "cpu", project=str(SCRATCH / "runs"), name=name,
                exist_ok=True, verbose=False, plots=False)
    best = YOLO(str(SCRATCH / "runs" / name / "weights" / "best.pt"))
    scores = {}
    for split in dict.fromkeys(["val", EVAL_SPLIT]):
        m = best.val(data=str(data_yaml), split=split, imgsz=IMAGE_SIZE, verbose=False, plots=False)
        scores[split] = {"map50": round(float(m.box.map50), 4), "map50_95": round(float(m.box.map), 4),
                         "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4)}
    onnx_path = best.export(format="onnx", imgsz=IMAGE_SIZE, opset=17, simplify=True)
    results.append({
        "model": name, "eval_split": EVAL_SPLIT,
        "eval_images": counts.get("test" if EVAL_SPLIT == "test" else "valid"),
        **scores[EVAL_SPLIT], "val": scores["val"],
        "train_minutes": round((time.time() - started) / 60, 1), "epochs": EPOCHS,
        "onnx": str(onnx_path), "onnx_mb": round(Path(onnx_path).stat().st_size / 1e6, 1),
    })
    print(results[-1])

In [ ]:
import hashlib

small, big = results[0], results[-1]
chosen = big if big["map50_95"] - small["map50_95"] >= MIN_GAIN_FOR_BIGGER_MODEL else small
reason = (f"{chosen['model']}: " + ("clearly more accurate" if chosen is big and big is not small
          else f"the larger model gains < {MIN_GAIN_FOR_BIGGER_MODEL:.2f} mAP50-95, and the small one is ~3x faster on CPU"))

shutil.copy(chosen["onnx"], OUT / "plate_detector.onnx")
sha = hashlib.sha256((OUT / "plate_detector.onnx").read_bytes()).hexdigest()
rows = "\n".join(
    f"| {r['model']} | {r['map50']:.3f} | {r['map50_95']:.3f} | {r['precision']:.3f} | {r['recall']:.3f} "
    f"| {r['val']['map50_95']:.3f} | {r['onnx_mb']} | {r['train_minutes']} |" for r in results)
sources_md = ""
if source_stats:
    sources_md = ("\n\n| Source | Licence | Role | Images kept | Duplicates | Plate crops dropped |\n"
                  "|---|---|---|---|---|---|\n" + "\n".join(
        f"| [{s['source']}](https://www.kaggle.com/datasets/{s['source']}) | {s['licence']} | {s['role']} "
        f"| {s['kept']} | {s['duplicates']} | {s['plate_crops']} |" for s in source_stats))
split_name = {"test": "test", "val": "validation"}[EVAL_SPLIT]
separate = " (a separate source, never used for training)" if DATASET_SOURCE == "kaggle_public" else ""
report = "\n".join([
    "# Plate detector: YOLOv8n vs YOLOv8s",
    "",
    f"Data: {source}; images per split {counts}.{sources_md}",
    "",
    f"Scored on the **{split_name}** split{separate}. {EPOCHS} epochs max (early stop patience 15), "
    f"image size {IMAGE_SIZE}, seed 0.",
    f"Trained on {gpu or 'CPU'} on {PLATFORM}, {time.strftime('%Y-%m-%d')}.",
    "",
    "| Model | mAP50 | mAP50-95 | Precision | Recall | Val mAP50-95 | ONNX (MB) | Train (min) |",
    "|---|---|---|---|---|---|---|---|",
    rows,
    "",
    f"**Chosen:** {reason}. `plate_detector.onnx` SHA-256 `{sha}`.",
    "CPU latency: run `eval/benchmark_detector.py models/plate_detector.onnx` on the gate machine.",
    "",
])
(OUT / "detector_comparison.md").write_text(report)
(OUT / "detector_comparison.json").write_text(json.dumps({
    "dataset": source, "platform": PLATFORM, "gpu": gpu, "splits": counts, "sources": source_stats,
    "results": results, "chosen": chosen["model"], "sha256": sha}, indent=2))
print(report)

In [ ]:
archive = shutil.make_archive(str(OUT.parent / "gatesense_detector_results"), "zip", OUT)
if ON_COLAB:
    from google.colab import files
    files.download(archive)
else:
    print("Done. Download", archive, "(Kaggle: the Output panel on the right, or the notebook's Output tab).")

## What to do with the files
The zip has `plate_detector.onnx`, `detector_comparison.md` and `.json`.

1. **Model → GitHub Release** (weights never go in git): on github.com/Akinchan-Jain-7/GateSense → *Releases* → *Draft a new release* → tag `models-v1` → attach `plate_detector.onnx` → *Publish*. Then anyone can run `uv run python -m gatesense_anpr.download_model --sha256 <hash from the report>`.
2. **Results → the repo:** put `detector_comparison.md` and `.json` in `anpr/eval/results/` (or give them to Claude to commit), so the README tables use these real numbers.

Licences: the training data is CC0 (public domain) and the test data Apache 2.0. The trained weights come from Ultralytics YOLOv8, which is AGPL-3.0 (see ADR-0009).